# Resolver — transparency pipeline eval notebook

Tests the `/resolve` endpoint. Stateless — no Weaviate, no fixture files.

**Pipeline (4 stages, all intra-document):**

| Stage | What | LLM? |
|-------|------|------|
| 1 · pre-merge | Normalize names; group same `(normalized_name, type)` into candidates | No |
| 2 · refine | Send candidates (per type) to LLM; skip if only 1 candidate per type | **Yes** (when ≥ 2 candidates of same type) |
| 3 · assign ids | Emit `ResolvedEntity` with sequential id and collected aliases | No |
| 4 · rewrite relations | Map head/tail names → ids; collapse overlap duplicates | No |

**Edge cases covered:**

| Case | What | LLM? |
|------|------|------|
| res-001 | empty input | No |
| res-002 | two distinct entities (1 per type) + relation — LLM skipped | No |
| res-003 | 3× same PERSON + legal-suffix ORG variant collapsed in pre-merge | No |
| res-004 | honorific stripped → two mentions become one candidate | No |
| res-005 | unknown relation endpoint → relation dropped | No |
| res-006 | overlap duplicate collapsed (same evidence fingerprint) | No |
| res-007 | distinct evidence → two separate relations kept | No |
| res-008 | definite description merged by LLM | **advisory** |
| res-009 | canonical name = longest raw form (short form arrives first) | No |
| res-010 | evidence strings from all pre-merged mentions collected | No |
| res-011 | `confidence = max` on overlap collapse (different confidences) | No |
| res-012 | non-empty relations but empty entities → all relations dropped | No |

**Scoring**: entity-level (canonical name + absorbed aliases + min evidence count) and
relation-level (head/tail resolved correctly, evidence count, min confidence, dropped count).

> Advisory cases require Ollama. If it is unavailable the service falls back to singletons
> and relations will still be written — advisory mismatches are informational.
>
> **Pronoun imputation is upstream.** `ner.coref.resolve_pronouns()` rewrites bare pronouns
> on raw text before NER extraction; by the time entities reach `/resolve` all pronouns are
> already gone. What the resolver handles (res-008) are leftover **definite descriptions**
> ("el concejal") that coref did not rewrite — not true pronouns.

In [ ]:
import os, sys, re, logging, unicodedata
from collections import defaultdict
import requests

NLP_BASE_URL = os.environ.get('NLP_BASE_URL', 'https://wiig.dia.fi.upm.es/nlp').rstrip('/')
ENDPOINT     = f'{NLP_BASE_URL}/resolve'
HEADERS      = {'Content-Type': 'application/json',
                'X-API-Key': os.environ.get('NLP_API_KEY', '')}
TIMEOUT      = 300   # LLM refine can take a while

logging.basicConfig(
    level=logging.INFO,
    format='%(name)-24s %(levelname)-8s  %(message)s',
    stream=sys.stdout, force=True,
)
for _noisy in ('httpx', 'httpcore', 'urllib3', 'requests', 'charset_normalizer'):
    logging.getLogger(_noisy).setLevel(logging.WARNING)
log = logging.getLogger('eval.resolver')

print(f'NLP  → {ENDPOINT}')

In [ ]:
# --- Service call -------------------------------------------------------

def call_resolve(entities, relations):
    r = requests.post(ENDPOINT,
                      json={'entities': entities, 'relations': relations},
                      headers=HEADERS, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()


# --- Pre-merge reconstruction (mirrors premerge.py, used for stage-1 trace) ---
_HONORIFIC = {'sr', 'sra', 'srta', 'dr', 'dra', 'don', 'dona', 'd', 'da', 'mr', 'mrs', 'ms'}
_LEGAL     = {'sa', 'sl', 'sau', 'slu', 'ltd', 'inc', 'gmbh', 'plc', 'llc', 'sas', 'bv', 'srl', 'co'}

def _normalize(name):
    s = ''.join(c for c in unicodedata.normalize('NFD', name.lower())
                if unicodedata.category(c) != 'Mn')
    s = re.sub(r'[.,;:"\' ()\[\]]', ' ', s)
    return ' '.join(t for t in s.split() if t and t not in _HONORIFIC and t not in _LEGAL)


def _estimate_candidates(entities):
    """Show which mentions collapse into which candidates (stage-1 trace)."""
    groups = defaultdict(list)
    for e in entities:
        groups[(_normalize(e['name']), e['type'])].append(e['name'])
    return groups


# --- Render + logging ---------------------------------------------------

def render(req_entities, req_relations, resp):
    out_entities  = resp.get('entities',  [])
    out_relations = resp.get('relations', [])

    # Stage 1 trace
    cands = _estimate_candidates(req_entities)
    n_in, n_cands = len(req_entities), len(cands)
    log.info('STAGE 1 · pre-merge  %d mentions → %d candidates', n_in, n_cands)
    for (norm, typ), names in cands.items():
        if len(names) > 1:
            log.info('  collapsed  type=%-14s  key=%r  names=%s', typ, norm, names)

    # Stage 2 trace (inferred)
    per_type = defaultdict(int)
    for (_, typ) in cands:
        per_type[typ] += 1
    llm_types = [t for t, n in per_type.items() if n >= 2]
    if llm_types:
        log.info('STAGE 2 · LLM called  types=%s', llm_types)
    else:
        log.info('STAGE 2 · LLM skipped (≤ 1 candidate per type)')

    # Stage 3: entities
    log.info('STAGE 3 · %d entities resolved', len(out_entities))
    print(f'  ENTITIES  ({len(out_entities)})')
    for e in out_entities:
        aliases = [n for n in e['names'] if n != e['canonical_name']]
        ali_s = f'  aliases={aliases}' if aliases else ''
        print(f"    [{e['id']}] {e['type']}  {e['canonical_name']!r}{ali_s}  "
              f"evidence_count={len(e['evidence'])}")

    # Stage 4: relations
    id_to_name = {e['id']: e['canonical_name'] for e in out_entities}
    n_in_rel   = len(req_relations)
    n_dropped  = n_in_rel - len(out_relations)
    log.info('STAGE 4 · %d/%d relations written  (%d dropped)',
             len(out_relations), n_in_rel, max(0, n_dropped))
    print(f'  RELATIONS ({len(out_relations)}/{n_in_rel}' +
          (f'  — {n_dropped} dropped' if n_dropped > 0 else '') + ')')
    for r in out_relations:
        h  = id_to_name.get(r['head_id'], r['head_id'])
        t  = id_to_name.get(r['tail_id'], r['tail_id'])
        ev = len(r.get('evidence', []))
        sub = f"  subtype={r['subtype']!r}" if r.get('subtype') else ''
        print(f"    {h!r} --[{r['type']}]--> {t!r}{sub}"
              f"  conf={r['confidence']:.2f}  evidence_count={ev}")


# --- Scoring ------------------------------------------------------------

def _name_match(a, b):
    return a.lower() in b.lower() or b.lower() in a.lower()


def _find_entity(canonical, etype, out_entities):
    for e in out_entities:
        if e['type'] == etype and _name_match(canonical, e['canonical_name']):
            return e
    return None


def score_entities(expected, out_entities):
    matched = 0
    for exp in expected:
        e = _find_entity(exp['canonical_name'], exp['type'], out_entities)
        if e is None:
            continue
        required = exp.get('names_include', [])
        if not all(any(_name_match(req, n) for n in e['names']) for req in required):
            continue
        if len(e.get('evidence', [])) < exp.get('min_evidence_count', 0):
            continue
        matched += 1
    return matched, len(expected)


def score_relations(expected, out_relations, out_entities):
    id_to_name = {e['id']: e['canonical_name'] for e in out_entities}
    matched = 0
    for exp in expected:
        for r in out_relations:
            h = id_to_name.get(r['head_id'], '')
            t = id_to_name.get(r['tail_id'], '')
            if (r['type'] == exp['type']
                    and _name_match(exp['head_name'], h)
                    and _name_match(exp['tail_name'], t)
                    and len(r.get('evidence', [])) >= exp.get('min_evidence', 1)
                    and r.get('confidence', 0) >= exp.get('min_confidence', 0.0)):
                matched += 1
                break
    return matched, len(expected)

In [ ]:
# ── CASES ─────────────────────────────────────────────────────────────────
# Domain: financial_flows — PERSON / ORGANIZATION entities, PAYMENT_TO relations.
#
# Each case has:
#   entities / relations           — the raw POST body
#   expected_entities              — list of {canonical_name, type, names_include?,
#                                             min_evidence_count?}
#   expected_entity_count          — exact count expected
#   expected_relations             — list of {head_name, tail_name, type, min_evidence?,
#                                             min_confidence?}
#   expected_relation_count        — exact count expected
#   advisory                       — True → LLM required; mismatch is informational

CASES = [
    # ── res-001 ── empty input ────────────────────────────────────────────
    {
        'id':          'res-001',
        'description': 'empty input — both lists empty → both empty out, no LLM',
        'entities':    [],
        'relations':   [],
        'expected_entity_count':   0,
        'expected_relation_count': 0,
        'expected_entities':       [],
        'expected_relations':      [],
    },

    # ── res-002 ── two unique entities, 1 per type → LLM skipped ─────────
    {
        'id':          'res-002',
        'description': 'two distinct entities (1 per type) + relation — LLM skipped',
        'entities': [
            {'name': 'Ana Martínez Ruiz', 'type': 'PERSON',
             'evidence': 'Ana Martínez Ruiz firmó el contrato de asesoría.'},
            {'name': 'Empresa Norte S.L.', 'type': 'ORGANIZATION',
             'evidence': 'Empresa Norte S.L. recibió el pago por los servicios prestados.'},
        ],
        'relations': [
            {'head': 'Ana Martínez Ruiz', 'tail': 'Empresa Norte S.L.',
             'type': 'PAYMENT_TO',
             'evidence': 'Ana Martínez Ruiz realizó un pago de 200.000 euros a Empresa Norte S.L.',
             'attributes': {'amount': 200000, 'currency': 'EUR'}, 'confidence': 0.92},
        ],
        'expected_entity_count':   2,
        'expected_relation_count': 1,
        'expected_entities': [
            {'canonical_name': 'Ana Martínez Ruiz',  'type': 'PERSON'},
            {'canonical_name': 'Empresa Norte S.L.', 'type': 'ORGANIZATION'},
        ],
        'expected_relations': [
            {'head_name': 'Ana Martínez', 'tail_name': 'Empresa Norte', 'type': 'PAYMENT_TO'},
        ],
        'notes': 'one PERSON + one ORGANIZATION → 1 candidate per type → _refine does not call Ollama',
    },

    # ── res-003 ── pre-merge: exact repeats + legal-suffix stripping ──────
    {
        'id':          'res-003',
        'description': 'repeated mentions (×3) + legal suffix variant — pre-merge collapses, no LLM',
        'entities': [
            {'name': 'Juan García López', 'type': 'PERSON',
             'evidence': 'Juan García López presidió la reunión de la comisión.'},
            {'name': 'Juan García López', 'type': 'PERSON',
             'evidence': 'Juan García López aprobó el presupuesto extraordinario.'},
            {'name': 'Juan García López', 'type': 'PERSON',
             'evidence': 'Juan García López firmó el contrato de adjudicación.'},
            {'name': 'Constructora del Sur SA', 'type': 'ORGANIZATION',
             'evidence': 'Constructora del Sur SA resultó adjudicataria del contrato.'},
            {'name': 'Constructora del Sur SL', 'type': 'ORGANIZATION',
             'evidence': 'Constructora del Sur SL recibió el primer pago en febrero.'},
        ],
        'relations': [
            {'head': 'Juan García López', 'tail': 'Constructora del Sur SA',
             'type': 'PAYMENT_TO',
             'evidence': 'Juan García López autorizó un pago de 450.000 euros a Constructora del Sur.',
             'attributes': {'amount': 450000, 'currency': 'EUR'}, 'confidence': 0.95},
        ],
        'expected_entity_count':   2,
        'expected_relation_count': 1,
        'expected_entities': [
            {'canonical_name': 'Juan García López', 'type': 'PERSON'},
            {'canonical_name': 'Constructora del Sur', 'type': 'ORGANIZATION',
             'names_include': ['Constructora del Sur SA', 'Constructora del Sur SL']},
        ],
        'expected_relations': [
            {'head_name': 'Juan García', 'tail_name': 'Constructora del Sur', 'type': 'PAYMENT_TO'},
        ],
        'notes': 'SA and SL (no dots) are in _LEGAL → both strip to "constructora del sur" → '
                 'same pre-merge key → 1 ORG candidate; 3× same PERSON → 1 PERSON candidate',
    },

    # ── res-004 ── pre-merge: honorific stripping ─────────────────────────
    {
        'id':          'res-004',
        'description': 'honorific prefix stripped in pre-merge — "Dr. García" ≡ "García"',
        'entities': [
            {'name': 'Dr. García Martínez', 'type': 'PERSON',
             'evidence': 'El Dr. García Martínez compareció ante la comisión de investigación.'},
            {'name': 'García Martínez', 'type': 'PERSON',
             'evidence': 'García Martínez declaró no conocer el origen de los fondos.'},
            {'name': 'Fundación Bienestar', 'type': 'ORGANIZATION',
             'evidence': 'La Fundación Bienestar recibió donaciones de origen no declarado.'},
        ],
        'relations': [
            {'head': 'Dr. García Martínez', 'tail': 'Fundación Bienestar',
             'type': 'PAYMENT_TO',
             'evidence': 'García Martínez realizó una donación anónima a la Fundación Bienestar.',
             'attributes': {}, 'confidence': 0.83},
        ],
        'expected_entity_count':   2,
        'expected_relation_count': 1,
        'expected_entities': [
            {'canonical_name': 'García Martínez', 'type': 'PERSON',
             'names_include': ['Dr. García Martínez', 'García Martínez']},
            {'canonical_name': 'Fundación Bienestar', 'type': 'ORGANIZATION'},
        ],
        'expected_relations': [
            {'head_name': 'García Martínez', 'tail_name': 'Fundación Bienestar', 'type': 'PAYMENT_TO'},
        ],
        'notes': '"dr" ∈ _HONORIFIC → stripped; both normalize to "garcia martinez" → same candidate',
    },

    # ── res-005 ── unknown endpoint → relation dropped ────────────────────
    {
        'id':          'res-005',
        'description': 'relation head not found in entities → that relation dropped; valid kept',
        'entities': [
            {'name': 'Ana Martínez', 'type': 'PERSON',
             'evidence': 'Ana Martínez suscribió el acuerdo de financiación.'},
            {'name': 'Inversiones Cenit SA', 'type': 'ORGANIZATION',
             'evidence': 'Inversiones Cenit SA gestionó el fondo de inversión.'},
        ],
        'relations': [
            {'head': 'Entidad Desconocida',   # not in entities → will be dropped
             'tail': 'Ana Martínez',
             'type': 'PAYMENT_TO',
             'evidence': 'Una entidad desconocida transfirió fondos a Ana Martínez.',
             'attributes': {}, 'confidence': 0.60},
            {'head': 'Ana Martínez', 'tail': 'Inversiones Cenit SA',
             'type': 'PAYMENT_TO',
             'evidence': 'Ana Martínez aportó 300.000 euros a Inversiones Cenit SA.',
             'attributes': {'amount': 300000, 'currency': 'EUR'}, 'confidence': 0.91},
        ],
        'expected_entity_count':   2,
        'expected_relation_count': 1,     # only the valid relation survives
        'expected_entities': [
            {'canonical_name': 'Ana Martínez',      'type': 'PERSON'},
            {'canonical_name': 'Inversiones Cenit', 'type': 'ORGANIZATION'},
        ],
        'expected_relations': [
            {'head_name': 'Ana Martínez', 'tail_name': 'Inversiones Cenit', 'type': 'PAYMENT_TO'},
        ],
        'notes': '"Entidad Desconocida" not in name_to_id → edges.py logs WARNING and drops it',
    },

    # ── res-006 ── overlap duplicate collapsed ────────────────────────────
    {
        'id':          'res-006',
        'description': 'same relation + identical evidence from two overlapping chunks → 1 relation, evidence_count=2',
        'entities': [
            {'name': 'Juan García', 'type': 'PERSON',
             'evidence': 'Juan García autorizó el pago.'},
            {'name': 'Constructora Norte', 'type': 'ORGANIZATION',
             'evidence': 'Constructora Norte recibió la transferencia.'},
        ],
        'relations': [
            {'head': 'Juan García', 'tail': 'Constructora Norte',
             'type': 'PAYMENT_TO',
             'evidence': 'Juan García autorizó una transferencia de 150.000 euros a Constructora Norte.',
             'attributes': {'amount': 150000, 'currency': 'EUR'}, 'confidence': 0.88},
            {'head': 'Juan García', 'tail': 'Constructora Norte',
             'type': 'PAYMENT_TO',
             'evidence': 'Juan García autorizó una transferencia de 150.000 euros a Constructora Norte.',
             'attributes': {'amount': 150000, 'currency': 'EUR'}, 'confidence': 0.88},
        ],
        'expected_entity_count':   2,
        'expected_relation_count': 1,     # duplicate collapsed
        'expected_entities': [
            {'canonical_name': 'Juan García',        'type': 'PERSON'},
            {'canonical_name': 'Constructora Norte', 'type': 'ORGANIZATION'},
        ],
        'expected_relations': [
            {'head_name': 'Juan García', 'tail_name': 'Constructora Norte',
             'type': 'PAYMENT_TO', 'min_evidence': 2},
        ],
        'notes': 'same (head_id, type, tail_id, evidence_fingerprint) → collapsed; '
                 'evidence list grows to 2; confidence = max(0.88, 0.88) = 0.88',
    },

    # ── res-007 ── distinct evidence = separate relations ─────────────────
    {
        'id':          'res-007',
        'description': 'same parties, two different payment events — not collapsed, both kept',
        'entities': [
            {'name': 'Roberto Vega', 'type': 'PERSON',
             'evidence': 'Roberto Vega gestionó los pagos del fondo.'},
            {'name': 'Shell Corp Ltd', 'type': 'ORGANIZATION',
             'evidence': 'Shell Corp Ltd domiciliada en las Islas Caimán.'},
        ],
        'relations': [
            {'head': 'Roberto Vega', 'tail': 'Shell Corp Ltd',
             'type': 'PAYMENT_TO',
             'evidence': 'Roberto Vega transfirió 80.000 euros a Shell Corp Ltd en enero de 2022.',
             'attributes': {'amount': 80000, 'currency': 'EUR'}, 'confidence': 0.90},
            {'head': 'Roberto Vega', 'tail': 'Shell Corp Ltd',
             'type': 'PAYMENT_TO',
             'evidence': 'Roberto Vega realizó un segundo pago de 120.000 euros a Shell Corp Ltd en julio de 2022.',
             'attributes': {'amount': 120000, 'currency': 'EUR'}, 'confidence': 0.90},
        ],
        'expected_entity_count':   2,
        'expected_relation_count': 2,     # distinct evidence → kept as two separate relations
        'expected_entities': [
            {'canonical_name': 'Roberto Vega',   'type': 'PERSON'},
            {'canonical_name': 'Shell Corp Ltd', 'type': 'ORGANIZATION'},
        ],
        'expected_relations': [
            {'head_name': 'Roberto Vega', 'tail_name': 'Shell Corp', 'type': 'PAYMENT_TO'},
            {'head_name': 'Roberto Vega', 'tail_name': 'Shell Corp', 'type': 'PAYMENT_TO'},
        ],
        'notes': 'different evidence fingerprints → different keys → 2 separate relations kept '
                 '(attributes are never aggregated: each payment is its own fact)',
    },

    # ── res-008 ── LLM merges definite description (advisory) ─────────────
    {
        'id':          'res-008',
        'description': 'definite description "el concejal" → LLM should merge into named PERSON  [advisory]',
        'entities': [
            {'name': 'Juan García López', 'type': 'PERSON',
             'evidence': 'Juan García López firmó el contrato de adjudicación directa.'},
            {'name': 'el concejal', 'type': 'PERSON',   # coref-leftover definite description
             'evidence': 'El concejal autorizó un pago de 450.000 euros a Constructora del Sur.'},
            {'name': 'Constructora del Sur', 'type': 'ORGANIZATION',
             'evidence': 'Constructora del Sur SL resultó adjudicataria.'},
        ],
        'relations': [
            {'head': 'el concejal', 'tail': 'Constructora del Sur',
             'type': 'PAYMENT_TO',
             'evidence': 'El concejal autorizó un pago de 450.000 euros a Constructora del Sur.',
             'attributes': {'amount': 450000, 'currency': 'EUR'}, 'confidence': 0.94},
        ],
        'expected_entity_count':   2,   # Juan + Constructora ("el concejal" merged into Juan)
        'expected_relation_count': 1,
        'expected_entities': [
            {'canonical_name': 'Juan García López', 'type': 'PERSON',
             'names_include': ['el concejal']},
            {'canonical_name': 'Constructora del Sur', 'type': 'ORGANIZATION'},
        ],
        'expected_relations': [
            {'head_name': 'Juan García', 'tail_name': 'Constructora', 'type': 'PAYMENT_TO'},
        ],
        'advisory': True,
        'notes': '"el concejal" and "Juan García López" differ after normalize → 2 PERSON candidates '
                 '→ LLM refine called → should merge. NB: this is a leftover definite description, '
                 'not a true pronoun — pronouns are resolved upstream by ner.coref before extraction.',
    },

    # ── res-009 ── canonical name = longest raw form ──────────────────────
    {
        'id':          'res-009',
        'description': 'canonical name = longest raw form — short form arrives first',
        'entities': [
            {'name': 'Inversora del Norte', 'type': 'ORGANIZATION',
             'evidence': 'Inversora del Norte presentó la documentación inicial.'},
            {'name': 'Inversora del Norte SL', 'type': 'ORGANIZATION',
             'evidence': 'Inversora del Norte SL firmó el contrato de adjudicación.'},
        ],
        'relations': [],
        'expected_entity_count':   1,
        'expected_relation_count': 0,
        'expected_entities': [
            {'canonical_name': 'Inversora del Norte SL', 'type': 'ORGANIZATION',
             'names_include': ['Inversora del Norte', 'Inversora del Norte SL']},
        ],
        'expected_relations': [],
        'notes': 'both normalize to "inversora del norte" (SL ∈ _LEGAL stripped); shorter form '
                 'arrives first; canonical = max(len) of raw names = "Inversora del Norte SL"',
    },

    # ── res-010 ── evidence collected from all pre-merged mentions ────────
    {
        'id':          'res-010',
        'description': 'evidence strings from all pre-merged mentions collected on resolved entity',
        'entities': [
            {'name': 'Carmen Blanco Rivas', 'type': 'PERSON',
             'evidence': 'Carmen Blanco Rivas compareció ante el tribunal.'},
            {'name': 'Carmen Blanco Rivas', 'type': 'PERSON',
             'evidence': 'Carmen Blanco Rivas negó haber recibido fondos.'},
            {'name': 'Carmen Blanco Rivas', 'type': 'PERSON',
             'evidence': 'Carmen Blanco Rivas firmó el acuerdo de no divulgación.'},
        ],
        'relations': [],
        'expected_entity_count':   1,
        'expected_relation_count': 0,
        'expected_entities': [
            {'canonical_name': 'Carmen Blanco Rivas', 'type': 'PERSON',
             'min_evidence_count': 3},
        ],
        'expected_relations': [],
        'notes': '3 identical-name mentions → 1 pre-merge candidate; '
                 'all 3 evidence strings must appear on the resolved entity',
    },

    # ── res-011 ── confidence = max on overlap collapse ───────────────────
    {
        'id':          'res-011',
        'description': 'overlap duplicate with different confidences → collapsed, confidence = max',
        'entities': [
            {'name': 'Pedro Ramos', 'type': 'PERSON',
             'evidence': 'Pedro Ramos autorizó el pago.'},
            {'name': 'Grupo Ibérico SA', 'type': 'ORGANIZATION',
             'evidence': 'Grupo Ibérico SA recibió los fondos.'},
        ],
        'relations': [
            {'head': 'Pedro Ramos', 'tail': 'Grupo Ibérico SA',
             'type': 'PAYMENT_TO',
             'evidence': 'Pedro Ramos transfirió 500.000 euros a Grupo Ibérico SA.',
             'attributes': {'amount': 500000, 'currency': 'EUR'}, 'confidence': 0.65},
            {'head': 'Pedro Ramos', 'tail': 'Grupo Ibérico SA',
             'type': 'PAYMENT_TO',
             'evidence': 'Pedro Ramos transfirió 500.000 euros a Grupo Ibérico SA.',
             'attributes': {'amount': 500000, 'currency': 'EUR'}, 'confidence': 0.95},
        ],
        'expected_entity_count':   2,
        'expected_relation_count': 1,     # duplicate collapsed
        'expected_entities': [
            {'canonical_name': 'Pedro Ramos',    'type': 'PERSON'},
            {'canonical_name': 'Grupo Ibérico SA', 'type': 'ORGANIZATION'},
        ],
        'expected_relations': [
            {'head_name': 'Pedro Ramos', 'tail_name': 'Grupo Ibérico', 'type': 'PAYMENT_TO',
             'min_confidence': 0.95},   # max(0.65, 0.95) = 0.95
        ],
        'notes': 'same evidence fingerprint → collapsed; confidence = max(0.65, 0.95) = 0.95',
    },

    # ── res-012 ── empty entities + non-empty relations → all dropped ─────
    {
        'id':          'res-012',
        'description': 'non-empty relations with empty entities → no IDs to rewrite → all dropped',
        'entities': [],
        'relations': [
            {'head': 'Alguien', 'tail': 'Alguna Empresa',
             'type': 'PAYMENT_TO',
             'evidence': 'Alguien realizó un pago a Alguna Empresa.',
             'attributes': {}, 'confidence': 0.80},
        ],
        'expected_entity_count':   0,
        'expected_relation_count': 0,     # no name_to_id entries → every endpoint lookup fails
        'expected_entities':  [],
        'expected_relations': [],
        'notes': 'entities=[] → name_to_id is empty → every head/tail lookup returns None → '
                 'all relations dropped at stage 4',
    },
]

print(f'{len(CASES)} cases')
for c in CASES:
    adv = '  [advisory]' if c.get('advisory') else ''
    print(f"  {c['id']}: {c['description']}{adv}")

In [ ]:
results = []   # (case_id, em, et, rm, rt, advisory, resp_or_None)

for case in CASES:
    adv_tag = ' [advisory]' if case.get('advisory') else ''
    print(f"── {case['id']}: {case['description']}{adv_tag}")
    try:
        resp = call_resolve(case['entities'], case['relations'])
    except Exception as exc:
        print(f'   ERROR: {exc}')
        if case.get('notes'):
            print(f'   note: {case["notes"]}')
        print()
        results.append((case['id'], 0, 1, 0, 1, case.get('advisory', False), None))
        continue

    render(case['entities'], case['relations'], resp)

    out_e = resp.get('entities',  [])
    out_r = resp.get('relations', [])

    # entity match
    em, et = score_entities(case['expected_entities'], out_e)
    ec_ok  = (len(out_e) == case['expected_entity_count'])

    # relation match
    rm, rt = score_relations(case['expected_relations'], out_r, out_e)
    rc_ok  = (len(out_r) == case['expected_relation_count'])

    passed = ec_ok and rc_ok and em == et and rm == rt
    icon   = '✅' if passed else ('⚠️ ' if case.get('advisory') else '❌')
    label  = 'pass' if passed else ('advisory mismatch' if case.get('advisory') else 'FAIL')
    print(f'   {icon} {label}')
    print(f'      entities  : {em}/{et} matched  count={len(out_e)}/{case["expected_entity_count"]}'
          + ('  ✅' if ec_ok and em==et else '  ❌'))
    print(f'      relations : {rm}/{rt} matched  count={len(out_r)}/{case["expected_relation_count"]}'
          + ('  ✅' if rc_ok and rm==rt else '  ❌'))
    if case.get('notes'):
        print(f'   note: {case["notes"]}')
    print()
    results.append((case['id'], em, et, rm, rt, case.get('advisory', False), resp))

In [ ]:
# ── SCORECARD ─────────────────────────────────────────────────────────────

hard = [(cid, em, et, rm, rt, resp) for cid, em, et, rm, rt, adv, resp in results if not adv]
adv  = [(cid, em, et, rm, rt, resp) for cid, em, et, rm, rt, adv, resp in results if adv]

def _case_passed(em, et, rm, rt, resp, expected_ec, expected_rc):
    if resp is None: return False
    return (em == et and rm == rt
            and len(resp.get('entities',  [])) == expected_ec
            and len(resp.get('relations', [])) == expected_rc)

case_map     = {c['id']: c for c in CASES}
hard_passed  = sum(1 for cid, em, et, rm, rt, resp in hard
                   if _case_passed(em, et, rm, rt, resp,
                                   case_map[cid]['expected_entity_count'],
                                   case_map[cid]['expected_relation_count']))
adv_passed   = sum(1 for cid, em, et, rm, rt, resp in adv
                   if _case_passed(em, et, rm, rt, resp,
                                   case_map[cid]['expected_entity_count'],
                                   case_map[cid]['expected_relation_count']))

total_em = sum(em for _, em, *_ in results)
total_et = sum(et for _, _, et, *_ in results)
total_rm = sum(rm for _, _, _, rm, *_ in results)
total_rt = sum(rt for _, _, _, _, rt, *_ in results)

print('SCORECARD · /resolve')
print(f'  Hard cases passing   : {hard_passed}/{len(hard)}')
print(f'  Advisory (LLM) cases : {adv_passed}/{len(adv)}')
print()
print(f'  Entities matched     : {total_em}/{total_et}')
print(f'  Relations matched    : {total_rm}/{total_rt}')
print()
print('PER-CASE')
print(f"  {'ID':<12}  {'Ent':>4}/{' ':>4}  {'Rel':>4}/{' ':>4}  status")
print('  ' + '-' * 52)
for cid, em, et, rm, rt, adv_flag, resp in results:
    exp_ec = case_map[cid]['expected_entity_count']
    exp_rc = case_map[cid]['expected_relation_count']
    passed = _case_passed(em, et, rm, rt, resp, exp_ec, exp_rc)
    icon   = '✅' if passed else ('⚠️ ' if adv_flag else '❌')
    got_ec = len(resp.get('entities',  [])) if resp else '?'
    got_rc = len(resp.get('relations', [])) if resp else '?'
    print(f'  {cid:<12}  {em:>4}/{et:<4}  {rm:>4}/{rt:<4}  {icon}'
          f'  entities={got_ec}/{exp_ec}  relations={got_rc}/{exp_rc}')